# DataMate — train baseline và phân tích lỗi

Notebook này **chạy inference trên mẫu phân tầng theo answer type của `data/train`**, sau đó chấm bằng `answers.jsonl`; không fine-tune model và không sửa source. Mặc định lấy 200/3.000 câu, rải ngẫu nhiên theo phân bố business trong từng loại đáp án. Kết quả ghi tại `/kaggle/working/train_baseline/`; có thể tải submission, logs và báo cáo xuống sau khi chạy.

PA thô trên mẫu phân tầng không đại diện trực tiếp cho PA toàn train, nên báo cáo có thêm PA ước lượng với trọng số theo tỷ lệ từng stratum trong train. Mẫu này dùng để chẩn đoán/so sánh vòng lặp phát triển, không thay thế validation độc lập.

Bật 2 GPU. Nếu đã chạy vLLM trước đó trong session, restart session trước khi chạy notebook để giải phóng VRAM. Sau khi sửa `src/` hoặc `prompts/`, đổi `RUN_TAG` và restart kernel/session rồi mới chạy lại; chỉ reload `predict.py` không reload các module con. Chạy các cell theo thứ tự.


In [ ]:
%pip install -q vllm==0.32.0


In [ ]:
%pip install --force-reinstall --no-deps torchaudio==2.11.0 --index-url https://download.pytorch.org/whl/cu130


In [ ]:
import torch
import torchaudio
from vllm import LLM, SamplingParams

print("PyTorch:", torch.__version__, "| CUDA:", torch.version.cuda)
print("TorchAudio:", torchaudio.__version__)
print("vLLM import OK; GPU count:", torch.cuda.device_count())
assert torch.version.cuda == "13.0", "TorchAudio wheel hiện được chọn cho CUDA 13.0."
assert torch.cuda.device_count() >= 2, "Notebook cần 2 GPU."


In [ ]:
from pathlib import Path
import os
import sys

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working/train_baseline")
SPLIT = "train"
RUN_TAG = "context_v5c_14b_int8_single"  # Một lần sinh code cho mỗi câu.
SAMPLE_N = 200
SEED = 42
MODEL_DIR = "Qwen/Qwen2.5-Coder-14B-Instruct-GPTQ-Int8"
MODEL_QUANTIZATION = "GPTQ-INT8"
TENSOR_PARALLEL_SIZE = 2
MAX_MODEL_LEN = 16384
GPU_MEMORY_UTILIZATION = 0.85
MAX_RETRIES = 2
ANALYSIS_PLAN_ENABLED = False  # Pilot mode; enable only for a separate benchmark run.
if ANALYSIS_PLAN_ENABLED:
    RUN_TAG += "_analysis_plan_pilot"

# Điền thủ công nếu Kaggle Input có nhiều bản code hoặc train.
CODE_DIR = None
DATA_DIR = None

def find_inputs(input_root):
    code_candidates = []
    train_candidates = []
    for root, dirs, files in os.walk(input_root):
        dirs[:] = [name for name in dirs if not name.startswith(".")]
        current = Path(root)
        if (current / "src" / "predict.py").is_file() and (current / "prompts").is_dir():
            code_candidates.append(current)
        required = {"questions.jsonl", "answers.jsonl", "calendar.csv", "rulebook.md"}
        if required.issubset(files) and (current / "data").is_dir():
            train_candidates.append(current)

    if CODE_DIR is None:
        if len(code_candidates) != 1:
            raise RuntimeError(
                f"Tìm thấy {len(code_candidates)} thư mục code; "
                "hãy đặt CODE_DIR thủ công."
            )
        code_path = code_candidates[0]
    else:
        code_path = Path(CODE_DIR)
    if DATA_DIR is None:
        preferred = [path for path in train_candidates if path.name.lower() == SPLIT]
        if not preferred:
            preferred = [path for path in train_candidates if SPLIT in path.parts]
        if len(preferred) != 1:
            raise RuntimeError(
                f"Tìm thấy {len(preferred)} thư mục {SPLIT}; "
                "hãy đặt DATA_DIR thủ công."
            )
        data_path = preferred[0]
    else:
        data_path = Path(DATA_DIR)

    if not (code_path / "src" / "predict.py").is_file():
        raise FileNotFoundError(f"CODE_DIR không hợp lệ: {code_path}")
    required_paths = [
        data_path / "questions.jsonl",
        data_path / "answers.jsonl",
        data_path / "calendar.csv",
        data_path / "rulebook.md",
        data_path / "data",
    ]
    missing = [str(path) for path in required_paths if not path.exists()]
    if missing:
        raise FileNotFoundError("Thiếu thành phần train: " + ", ".join(missing))
    return code_path.resolve(), data_path.resolve()

code_path, data_path = find_inputs(INPUT_ROOT)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Code:", code_path)
print("Train:", data_path)
print("Output:", OUTPUT_ROOT)


In [ ]:
import json
from collections import Counter, defaultdict
import numpy as np

with (data_path / "questions.jsonl").open(encoding="utf-8") as file:
    all_questions = [json.loads(line) for line in file if line.strip()]
with (data_path / "answers.jsonl").open(encoding="utf-8") as file:
    all_answers = [json.loads(line) for line in file if line.strip()]

question_ids = [record["id"] for record in all_questions]
answer_ids = [record["id"] for record in all_answers]
if len(question_ids) != len(set(question_ids)) or len(answer_ids) != len(set(answer_ids)):
    raise ValueError("ID câu hỏi hoặc đáp án bị trùng.")
if set(question_ids) != set(answer_ids):
    raise ValueError("questions.jsonl và answers.jsonl không có cùng tập ID.")
if not 1 <= SAMPLE_N <= len(all_questions):
    raise ValueError(f"SAMPLE_N phải từ 1 đến {len(all_questions)}.")

answers_by_id = {answer["id"]: answer for answer in all_answers}
type_buckets = defaultdict(list)
for question in all_questions:
    answer_type = answers_by_id[question["id"]].get("type", "unknown")
    type_buckets[answer_type].append(question)

def allocate_proportional_quotas(populations, sample_size):
    total_population = sum(populations.values())
    exact = {key: sample_size * count / total_population for key, count in populations.items()}
    quotas = {key: min(count, int(exact[key])) for key, count in populations.items()}
    remaining = sample_size - sum(quotas.values())
    order = sorted(
        populations,
        key=lambda key: (-(exact[key] - int(exact[key])), str(key)),
    )
    while remaining:
        progressed = False
        for key in order:
            if quotas[key] < populations[key]:
                quotas[key] += 1
                remaining -= 1
                progressed = True
                if remaining == 0:
                    break
        if not progressed:
            raise ValueError("Không đủ câu hỏi để tạo sample với kích thước yêu cầu.")
    return quotas

type_populations = {answer_type: len(rows) for answer_type, rows in type_buckets.items()}
type_quotas = allocate_proportional_quotas(type_populations, SAMPLE_N)
rng = np.random.default_rng(SEED)
sampled_questions = []
for answer_type, rows in type_buckets.items():
    indices = rng.choice(len(rows), size=type_quotas[answer_type], replace=False)
    sampled_questions.extend(rows[int(index)] for index in indices)

sampled_questions.sort(key=lambda question: (question["business_id"], question["id"]))
sampled_ids = {question["id"] for question in sampled_questions}
sampled_answers = {question_id: answers_by_id[question_id] for question_id in sampled_ids}

import hashlib
import re
from datetime import datetime, timezone

if not re.fullmatch(r"[A-Za-z0-9._-]+", RUN_TAG):
    raise ValueError("RUN_TAG chỉ được chứa chữ, số, dấu chấm, gạch dưới hoặc gạch ngang.")
RUN_DIR = OUTPUT_ROOT / f"{RUN_TAG}_seed_{SEED}_n_{SAMPLE_N}"
RUN_DIR.mkdir(parents=True, exist_ok=True)
code_digest = hashlib.sha256()
for source_file in sorted(list((code_path / "src").rglob("*.py")) + list((code_path / "prompts").rglob("*"))):
    if source_file.is_file():
        code_digest.update(str(source_file.relative_to(code_path)).encode("utf-8"))
        code_digest.update(source_file.read_bytes())
data_digest = hashlib.sha256()
data_files = [data_path / "calendar.csv", data_path / "rulebook.md"]
if (data_path / "README.md").is_file():
    data_files.append(data_path / "README.md")
data_files.extend(path for path in (data_path / "data").rglob("*") if path.is_file())
for source_file in sorted(data_files):
    data_digest.update(str(source_file.relative_to(data_path)).encode("utf-8"))
    data_digest.update(source_file.read_bytes())
sample_signature = hashlib.sha256(
    "\n".join(sorted(sampled_ids)).encode("utf-8")
).hexdigest()
run_config = {
    "run_tag": RUN_TAG,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "split": SPLIT,
    "sample_n": SAMPLE_N,
    "seed": SEED,
    "sample_signature": sample_signature,
    "model": MODEL_DIR,
    "model_quantization": MODEL_QUANTIZATION,
    "tensor_parallel_size": TENSOR_PARALLEL_SIZE,
    "max_model_len": MAX_MODEL_LEN,
    "prompt_token_budget": MAX_MODEL_LEN - 1024 - 512,
    "gpu_memory_utilization": GPU_MEMORY_UTILIZATION,
    "max_retries": MAX_RETRIES,
    "confidence_threshold": 0.35,
    "analysis_plan_enabled": ANALYSIS_PLAN_ENABLED,
    "code_sha256": code_digest.hexdigest(),
    "business_data_sha256": data_digest.hexdigest(),
    "questions_sha256": hashlib.sha256((data_path / "questions.jsonl").read_bytes()).hexdigest(),
    "answers_sha256": hashlib.sha256((data_path / "answers.jsonl").read_bytes()).hexdigest(),
}
config_path = RUN_DIR / "run_config.json"
if config_path.exists():
    old_config = json.loads(config_path.read_text(encoding="utf-8"))
    comparable_old = {key: value for key, value in old_config.items() if key != "created_utc"}
    comparable_new = {key: value for key, value in run_config.items() if key != "created_utc"}
    if comparable_old != comparable_new:
        raise ValueError(
            f"RUN_TAG={RUN_TAG!r} đã tồn tại với code/dữ liệu/cấu hình khác. "
            "Đổi RUN_TAG để tạo run độc lập; giữ nguyên tag chỉ để resume cùng run."
        )
else:
    legacy_artifacts = [
        RUN_DIR / name for name in ("submission.jsonl", "diagnostics.jsonl", "sample_questions.jsonl")
    ]
    if any(path.exists() for path in legacy_artifacts):
        raise ValueError(
            "RUN_DIR có kết quả cũ nhưng thiếu run_config.json; đổi RUN_TAG để không dùng nhầm kết quả."
        )
    config_path.write_text(json.dumps(run_config, ensure_ascii=False, indent=2), encoding="utf-8")
SAMPLE_PATH = RUN_DIR / "sample_questions.jsonl"

if SAMPLE_PATH.exists():
    existing_sample = [json.loads(line) for line in SAMPLE_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
    if [question["id"] for question in existing_sample] != [question["id"] for question in sampled_questions]:
        raise ValueError("Sample manifest trong RUN_DIR không khớp cấu hình hiện tại.")
else:
    SAMPLE_PATH.write_text(
        "".join(json.dumps(q, ensure_ascii=False) + "\n" for q in sampled_questions),
        encoding="utf-8",
    )

print(f"Training questions: {len(all_questions)} | answers: {len(all_answers)}")
print(f"Sample: {len(sampled_questions)} | answer types represented: {len(type_quotas)}")
print("Train answer-type distribution:", dict(Counter(a.get("type", "unknown") for a in all_answers)))
print("Sample answer-type distribution:", dict(Counter(a.get("type", "unknown") for a in sampled_answers.values())))
print("Sample business distribution:", dict(Counter(q["business_id"] for q in sampled_questions)))
print("Run config:", config_path)
print("Saved sample manifest:", SAMPLE_PATH)


In [ ]:
import importlib
import inspect
import json
import logging
import time
from datetime import date

importlib.invalidate_caches()
for module_name in tuple(sys.modules):
    if (
        module_name == "src"
        or module_name.startswith("src.")
        or module_name == "modules"
        or module_name.startswith("modules.")
    ):
        del sys.modules[module_name]
sys.path = [entry for entry in sys.path if Path(entry or ".").resolve() != code_path]
sys.path.insert(0, str(code_path))
import src.predict as predict_module
predict_module = importlib.reload(predict_module)
required_signatures = {
    "DataMatePipeline.__init__": (
        predict_module.DataMatePipeline.__init__,
        {
            "prompt_token_counter",
            "max_prompt_tokens",
        },
    ),
    "PromptBuilder.build": (
        predict_module.PromptBuilder.build,
        {"token_counter", "max_tokens"},
    ),
    "RetryEngine.run": (
        predict_module.RetryEngine.run,
        {"token_counter", "max_prompt_tokens"},
    ),
}
incompatible_signatures = {}
for symbol, (function, required_parameters) in required_signatures.items():
    signature = inspect.signature(function)
    missing = required_parameters - set(signature.parameters)
    if missing:
        incompatible_signatures[symbol] = {
            "missing": sorted(missing),
            "signature": str(signature),
        }
print("Imported predict.py:", Path(predict_module.__file__).resolve())
prompt_builder_module = sys.modules[predict_module.PromptBuilder.__module__]
print("Imported prompt_builder.py:", Path(prompt_builder_module.__file__).resolve())
retry_engine_module = sys.modules[predict_module.RetryEngine.__module__]
print("Imported retry_engine.py:", Path(retry_engine_module.__file__).resolve())
for module_name, module in {
    "predict.py": predict_module,
    "prompt_builder.py": prompt_builder_module,
    "retry_engine.py": retry_engine_module,
}.items():
    module_path = Path(module.__file__).resolve()
    if code_path.resolve() not in module_path.parents:
        incompatible_signatures[module_name] = {
            "unexpected_path": str(module_path),
            "expected_under": str(code_path.resolve()),
        }
if incompatible_signatures:
    raise RuntimeError(
        "DataMate import preflight failed. A stale `modules.*` cache or a "
        "mixed source directory may be active. Inspect the module paths printed "
        f"above and signature details: {incompatible_signatures}. This notebook "
        "now clears both `src.*` and `modules.*` caches before importing. If any "
        "path is outside CODE_DIR, correct CODE_DIR or restart the Kaggle session."
    )
from src.modules.answer_formatter import build_submission_line

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler(RUN_DIR / "run.log", encoding="utf-8"),
    ],
    force=True,
)

if torch.cuda.device_count() < TENSOR_PARALLEL_SIZE:
    raise RuntimeError(f"Cần {TENSOR_PARALLEL_SIZE} GPU.")

# This checkpoint's config.json declares GPTQ with 8-bit weights; vLLM auto-detects it.
llm = LLM(
    model=MODEL_DIR,
    tensor_parallel_size=TENSOR_PARALLEL_SIZE,
    trust_remote_code=True,
    max_model_len=MAX_MODEL_LEN,
    gpu_memory_utilization=GPU_MEMORY_UTILIZATION,
    dtype="float16",
)
tokenizer = llm.get_tokenizer()
sampling_params = SamplingParams(
    temperature=0.0,
    max_tokens=1024,
    stop=["```\n", "\n\n\n"],
    include_stop_str_in_output=True,
)

current_question_id = None
attempt_history = defaultdict(list)

def generate_with_vllm(prompt):
    if getattr(tokenizer, "chat_template", None):
        prompt = tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt}],
            tokenize=False,
            add_generation_prompt=True,
        )
    try:
        return llm.generate([prompt], sampling_params, use_tqdm=False)[0].outputs[0].text
    except Exception as exc:
        if current_question_id is not None:
            attempt_history[current_question_id].append({
                "phase": "generation",
                "success": False,
                "error": f"{type(exc).__name__}: {exc}",
            })
        raise

def count_prompt_tokens(prompt):
    if getattr(tokenizer, "chat_template", None):
        prompt = tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt}],
            tokenize=False,
            add_generation_prompt=True,
        )
    return len(tokenizer.encode(prompt, add_special_tokens=False))

pipeline = predict_module.DataMatePipeline(
    data_dir=str(data_path),
    output_dir=str(RUN_DIR),
    model_dir=MODEL_DIR,
    max_retries=MAX_RETRIES,
    confidence_threshold=0.35,
    llm_generate_fn=generate_with_vllm,
    prompt_token_counter=count_prompt_tokens,
    max_prompt_tokens=MAX_MODEL_LEN - 1024 - 512,
    analysis_plan_enabled=ANALYSIS_PLAN_ENABLED,
)

_original_execute = pipeline.code_executor.execute
def recording_execute(*args, **kwargs):
    try:
        result = _original_execute(*args, **kwargs)
    except Exception as exc:
        if current_question_id is not None:
            attempt_history[current_question_id].append({
                "phase": "execution",
                "success": False,
                "error": f"{type(exc).__name__}: {exc}",
                "generated_code": "",
            })
        raise
    if current_question_id is not None:
        attempt_history[current_question_id].append({
            "phase": "execution",
            "success": result.success,
            "error": result.error,
            "generated_code": result.code,
        })
    return result
pipeline.code_executor.execute = recording_execute

submission_path = RUN_DIR / "submission.jsonl"
diagnostics_path = RUN_DIR / "diagnostics.jsonl"
existing_predictions = {}
if submission_path.exists():
    for line in submission_path.read_text(encoding="utf-8").splitlines():
        if line.strip():
            record = json.loads(line)
            question_id = record["id"]
            if question_id not in sampled_ids or question_id in existing_predictions:
                raise ValueError(f"Submission cũ có ID lạ/trùng: {question_id}")
            existing_predictions[question_id] = record

existing_diagnostics = {}
if diagnostics_path.exists():
    for line in diagnostics_path.read_text(encoding="utf-8").splitlines():
        if line.strip():
            record = json.loads(line)
            existing_diagnostics[record["id"]] = record

with submission_path.open("a", encoding="utf-8") as submission_file, diagnostics_path.open(
    "a", encoding="utf-8"
) as diagnostics_file:
    pending = [q for q in sampled_questions if q["id"] not in existing_predictions]
    for index, question in enumerate(pending, start=1):
        question_id = question["id"]
        current_question_id = question_id
        attempt_history[question_id] = []
        started = time.time()
        expected_type = None
        try:
            expected_type = pipeline.question_analyzer.analyze(
                question["question"], date.fromisoformat(question["as_of_date"])
            ).expected_type
            answer = pipeline.process_question(question)
            line = build_submission_line(question_id, answer)
            record = json.loads(line)
        except Exception as exc:
            logging.exception("Question %s failed; writing abstain.", question_id)
            record = {"id": question_id, "type": "abstain"}
            line = json.dumps(record, ensure_ascii=False)
            attempt_history[question_id].append({
                "phase": "pipeline",
                "success": False,
                "error": f"{type(exc).__name__}: {exc}",
            })

        submission_file.write(line + "\n")
        submission_file.flush()
        existing_predictions[question_id] = record
        diagnostic = {
            "id": question_id,
            "business_id": question["business_id"],
            "question": question["question"],
            "reference_type": sampled_answers[question_id].get("type"),
            "expected_type": expected_type,
            "prediction_type": record.get("type"),
            "attempts": attempt_history[question_id],
            "elapsed_seconds": round(time.time() - started, 2),
        }
        diagnostics_file.write(json.dumps(diagnostic, ensure_ascii=False) + "\n")
        diagnostics_file.flush()
        existing_diagnostics[question_id] = diagnostic
        current_question_id = None
        print(f"[{len(existing_predictions)}/{len(sampled_questions)}] {question_id}: {record.get('type')}")

print("Submission:", submission_path)
print("Diagnostics:", diagnostics_path)
print("Per-question logs:", RUN_DIR / "logs")


In [ ]:
import json
from collections import Counter, defaultdict
import pandas as pd
from src.evaluate import compute_pa

predictions = {
    question_id: existing_predictions[question_id]
    for question_id in sampled_ids
    if question_id in existing_predictions
}
metrics = compute_pa(predictions, sampled_answers)
scores_by_id = {item["id"]: item["score"] for item in metrics["details"]}
sampled_type_counts = Counter(
    answer.get("type", "unknown") for answer in sampled_answers.values()
)
population_type_counts = Counter(
    answer.get("type", "unknown") for answer in all_answers
)

weighted_score = 0.0
weighted_type_scores = defaultdict(float)
for question in sampled_questions:
    question_id = question["id"]
    answer_type = sampled_answers[question_id].get("type", "unknown")
    weight = population_type_counts[answer_type] / sampled_type_counts[answer_type]
    score = scores_by_id[question_id]
    weighted_score += weight * score
    weighted_type_scores[answer_type] += weight * score
weighted_pa = weighted_score / len(all_questions)

diagnostic_rows = []
for question in sampled_questions:
    question_id = question["id"]
    diagnostic = existing_diagnostics.get(question_id, {})
    prediction = predictions.get(question_id, {})
    reference = sampled_answers[question_id]
    attempts = diagnostic.get("attempts", [])
    errors = [attempt.get("error") for attempt in attempts if attempt.get("error")]
    first_error = str(errors[0]).splitlines()[0] if errors else ""
    prediction_type = prediction.get("type", "missing")
    reference_type = reference.get("type", "unknown")
    score = scores_by_id[question_id]

    if score == 1:
        category = "correct"
    elif prediction_type == "abstain":
        category = "abstain_exec_error" if errors else "abstain_gate"
    elif prediction_type == "unanswerable" and reference_type != "unanswerable":
        category = "false_unanswerable"
    elif reference_type == "unanswerable" and prediction_type != "unanswerable":
        category = "missed_unanswerable"
    elif prediction_type != reference_type:
        category = "wrong_type"
    else:
        category = "wrong_value"

    log_path = RUN_DIR / "logs" / f"{question_id}.json"
    log_entry = json.loads(log_path.read_text(encoding="utf-8")) if log_path.exists() else {}
    diagnostic_rows.append({
        "id": question_id,
        "business_id": question["business_id"],
        "question": question["question"],
        "pred_answer": prediction.get("answer"),
        "ref_answer": reference.get("answer"),
        "reason": prediction.get("reason"),
        "reference_type": reference_type,
        "expected_type": diagnostic.get("expected_type"),
        "prediction_type": prediction_type,
        "score": score,
        "type_weight": population_type_counts[reference_type] / sampled_type_counts[reference_type],
        "retries": log_entry.get("retries"),
        "elapsed_seconds": diagnostic.get("elapsed_seconds"),
        "error_category": category,
        "error": " | ".join(str(error) for error in errors),
        "attempts": json.dumps(attempts, ensure_ascii=False),
        "generated_code": log_entry.get("generated_code", ""),
    })

diagnostics_df = pd.DataFrame(diagnostic_rows)
diagnostics_df.to_csv(RUN_DIR / "error_analysis.csv", index=False, encoding="utf-8-sig")
valid_type_rows = [
    row for row in diagnostic_rows
    if row["expected_type"] is not None and row["reference_type"] != "unanswerable"
]
total_elapsed = sum(row["elapsed_seconds"] or 0 for row in diagnostic_rows)
elapsed_by_type = defaultdict(list)
for row in diagnostic_rows:
    if row["elapsed_seconds"] is not None:
        elapsed_by_type[row["reference_type"]].append(row["elapsed_seconds"])

report = {
    "sample_size": len(sampled_questions),
    "train_size": len(all_questions),
    "seed": SEED,
    "run_tag": RUN_TAG,
    "unweighted_sample_pa": metrics["pa"],
    "weighted_train_pa_estimate": round(weighted_pa, 4),
    "correct": metrics["correct"],
    "wrong": metrics["wrong"],
    "abstain": metrics["abstain"],
    "missing": metrics["missing"],
    "per_reference_type_sample": metrics["per_type"],
    "per_reference_type_weighted_pa_estimate": {
        answer_type: round(weighted_type_scores[answer_type] / population_type_counts[answer_type], 4)
        for answer_type in population_type_counts
        if population_type_counts[answer_type]
    },
    "error_categories": dict(Counter(row["error_category"] for row in diagnostic_rows)),
    "expected_type_accuracy_excluding_unanswerable": round(
        sum(row["expected_type"] == row["reference_type"] for row in valid_type_rows)
        / len(valid_type_rows),
        4,
    ) if valid_type_rows else None,
    "elapsed_seconds_total": round(total_elapsed, 2),
    "elapsed_seconds_mean_per_question": round(total_elapsed / len(diagnostic_rows), 2)
    if diagnostic_rows else None,
    "elapsed_seconds_mean_by_reference_type": {
        answer_type: round(sum(values) / len(values), 2)
        for answer_type, values in elapsed_by_type.items() if values
    },
}
(RUN_DIR / "eval_summary.json").write_text(
    json.dumps(report, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(json.dumps(report, ensure_ascii=False, indent=2))
print("Detailed diagnostics:", RUN_DIR / "error_analysis.csv")
